In [2]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm
import requests
import libpysal
from esda.moran import Moran
from esda.smoothing import Empirical_Bayes
import gerrychain
from gerrychain import Graph
from libpysal.weights import Queen
import matplotlib.gridspec as gridspec
import textwrap


from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegressionCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss, accuracy_score

In [3]:
census_and_geodata = pd.read_parquet('nation_census_and_geodata.parquet')
vars_no_log = ['MEDAGE', 'VOTELEAN', 'INTPTLON']
vars_log = ['DENSITY', 'BACHDEG', 'SPANISHLIMENGLISH', 'FOREIGNBORN', 'MEXICANORIGIN','SOUTHAMORIGIN', 'CENTRALAMORIGIN', 'CARIBBEAN', 'NOINTERNET', "RENTERS", "NONCITIZENS", 'MEDINCOME']


# Binary predictor-- not strong

In [7]:
# ==========================================
# 6.A LOGISTIC REGRESSION & PREDICTIVE MODEL
# ==========================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegressionCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss, accuracy_score


# 1. Reconstruct feature dataframe based on your lists
X_features = census_and_geodata[vars_no_log].copy()

for col in vars_log:
    # Using log1p (log(1+x)) keeps 0 entries safe from evaluating to negative infinity
    X_features[f'log_{col}'] = np.log1p(census_and_geodata[col])

# Choose your target column (e.g., 'Most_SOR')
y_target = census_and_geodata['Most_SOR']

# Combine to perform an explicit dropna alignment across rows
model_data = pd.concat([X_features, y_target], axis=1).dropna()

# Final clean feature matrix and target vector
X = model_data.drop(columns=['Most_SOR'])
y = model_data['Most_SOR'].astype(int)

# Train-test split (80/20 ratio matching organ-followup standard)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Calculate naive mean probability from training target
mean_prob = y_train.mean()
y_train_baseline = np.full_like(y_train, mean_prob, dtype=float)
y_test_baseline = np.full_like(y_test, mean_prob, dtype=float)

# Determine global majority class for accuracy baseline
majority_class = 1 if mean_prob >= 0.5 else 0

print("--- Baseline Naive Model ---")
print(f"Train Log Loss: {log_loss(y_train, y_train_baseline):.4f}")
print(f"Test Log Loss:  {log_loss(y_test, y_test_baseline):.4f}")
print(f"Train Accuracy: {accuracy_score(y_train, np.full_like(y_train, majority_class)):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, np.full_like(y_test, majority_class)):.4f}")

# Build a pipeline to scale data uniformly across folds
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr_cv', LogisticRegressionCV(penalty='l1', solver='saga', cv=5, max_iter=5000, random_state=42))
])

# Train model
lr_pipeline.fit(X_train, y_train)

print("--- Regularized Logistic Regression (L1-CV) ---")
print(f"Train Log Loss: {log_loss(y_train, lr_pipeline.predict_proba(X_train)):.4f}")
print(f"Test Log Loss:  {log_loss(y_test, lr_pipeline.predict_proba(X_test)):.4f}")
print(f"Train Accuracy: {accuracy_score(y_train, lr_pipeline.predict(X_train)):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, lr_pipeline.predict(X_test)):.4f}")


# Initialize Random Forest (max_depth restricted to mitigate severe overfitting)
rf_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf_model.fit(X_train, y_train)

print("--- Random Forest Classifier ---")
print(f"Train Log Loss: {log_loss(y_train, rf_model.predict_proba(X_train)):.4f}")
print(f"Test Log Loss:  {log_loss(y_test, rf_model.predict_proba(X_test)):.4f}")
print(f"Train Accuracy: {accuracy_score(y_train, rf_model.predict(X_train)):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, rf_model.predict(X_test)):.4f}")

# Create a DataFrame matching features to coefficients, for the logistic regression 
importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': lr_pipeline.named_steps['lr_cv'].coef_[0]
})

# Sort by absolute magnitude to see the most influential features at the top
importance_df['Abs_Magnitude'] = importance_df['Coefficient'].abs()
importance_df = importance_df.sort_values(by='Abs_Magnitude', ascending=False).drop(columns=['Abs_Magnitude'])

# Display the clean dataframe
print('--- Logistic regression importance dataframe ---')
print(importance_df.reset_index(drop=True))


# Create a DataFrame matching features to coefficients, for the Random Forest
rf_importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_model.feature_importances_
})

rf_importance_df = rf_importance_df.sort_values(
    by='Importance',
    ascending=False
).reset_index(drop=True)

print('--- Random Forest importance dataframe ---')
rf_importance_df



/Users/ireland/Desktop/Spring2026/.venv/lib/python3.11/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/ireland/Desktop/Spring2026/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/Users/ireland/Desktop/Spring2026/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1811: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' instead. Use l1_ratios=(0,) instead of penalty='l2'  and l1_ratios=(1,) instead of penalty='l1'.
  warnings.warn(
/Users/ire

--- Baseline Naive Model ---
Train Log Loss: 0.6907
Test Log Loss:  0.6822
Train Accuracy: 0.5348
Test Accuracy:  0.5962
--- Regularized Logistic Regression (L1-CV) ---
Train Log Loss: 0.5724
Test Log Loss:  0.5648
Train Accuracy: 0.7159
Test Accuracy:  0.7122
--- Random Forest Classifier ---
Train Log Loss: 0.3763
Test Log Loss:  0.5067
Train Accuracy: 0.8806
Test Accuracy:  0.7552
--- Logistic regression importance dataframe ---
                  Feature  Coefficient
0             log_BACHDEG    -0.987703
1   log_SPANISHLIMENGLISH     0.849609
2         log_NONCITIZENS     0.784204
3     log_CENTRALAMORIGIN     0.635723
4         log_FOREIGNBORN    -0.488794
5             log_DENSITY     0.252535
6                  MEDAGE    -0.219798
7          log_NOINTERNET     0.207099
8       log_SOUTHAMORIGIN    -0.204610
9       log_MEXICANORIGIN    -0.138868
10            log_RENTERS     0.137586
11          log_CARIBBEAN    -0.118767
12               INTPTLON     0.096417
13               VO

,Feature,Importance
0,log_SPANISHLIMENGLISH,0.155074
1,log_MEXICANORIGIN,0.100223
2,log_CENTRALAMORIGIN,0.098299
3,log_NONCITIZENS,0.085742
4,log_BACHDEG,0.076906
5,MEDAGE,0.074569
6,INTPTLON,0.059553
7,log_FOREIGNBORN,0.053022
8,log_RENTERS,0.052071
9,log_MEDINCOME,0.047983


### Predictive Model Evaluation & Feature Analysis: Some Other Race Alone

Following our predictive modeling pipeline, we evaluated models targeting `Most_SOR`. We have a training mean of 53.48%, which indicates we have a highly balanced classification boundary (the baseline model is not depending on one class over another).

#### 1. Performance Metrics Summary

| Model | Train Log Loss | Test Log Loss | Train Accuracy | Test Accuracy |
| :--- | :---: | :---: | :---: | :---: |
| **Baseline Naive Model** | 0.6907 | 0.6822 | 53.48% | 59.62% |
| **Regularized Logistic Regression ($L_1$-CV)** | 0.5724 | 0.5648 | 71.59% | 71.22% |
| **Random Forest Classifier** | 0.3763 | 0.5067 | 88.06% | 75.52% |

---

#### 2. Key Findings & Performance Interpretation

* Logistic Regression improved accuracy (in comparison to the baseline) by over 11.6%, and the Random Forest improved the baseline by 15.9%. This confirms that we are capturing signals.

* Logistic Regression: The Logistic Regression model demonstrates exceptional generalization, as the train and test accuracy are very close. Log Losses are similarly close. This indicates that the lasso penalty successfully selected robust linear features without memorizing specifics to the training data.

* Random Forest: This model achieved the highest overall predictive accuracy on the test set; however, unlike with the logistic regression, we see a drop-off from its training accuracy. This suggests some mild overfitting.

---

#### 3. Feature Importance: Logistic Regression

The coefficients will tell us about how each variable impacts SOR being the most popular choice. We sort by the absolute value of the coefficient, as this tells us the variables that have the biggest effect.

| Feature Name | Coefficient Value | Impact Direction on `Most_SOR` |
| :--- | :---: | :--- |
| **`log_BACHDEG`** | `-0.987703` | **Strong Negative:** The single most dominant negative predictor in the multivariate model. Higher rates of college graduation strongly decrease a county's likelihood of being `Most_SOR` when other factors are controlled. |
| **`log_SPANISHLIMENGLISH`** | `+0.849609` | **Strong Positive:** Higher proportions of households with limited English proficiency act as the strongest positive anchor driving the classification upward. |
| **`log_NONCITIZENS`** | `+0.784204` | **Strong Positive:** Higher percentages of non-citizen residents serve as a primary upward marker for the target. |
| **`log_CENTRALAMORIGIN`** | `+0.635723` | **Moderate-Strong Positive:** Higher concentrations of Central American ancestry act as a key descriptive positive qualifier in the combined model. |
| **`log_FOREIGNBORN`** | `-0.488794` | **Moderate Negative:** Holds a moderate negative predictive weight in the combined model, adjusting baseline immigrant distributions. |
| **`log_DENSITY`** | `+0.252535` | **Moderate Positive:** Higher urban population density increases the odds of matching this profile. |
| **`MEDAGE`** | `-0.219798` | **Moderate Negative:** Counties with older median ages shift predictions marginally away from `Most_SOR`. |
| **`log_NOINTERNET`** | `+0.207099` | **Moderate Positive:** Areas with higher digital infrastructure deficits correlate with higher odds of meeting the threshold. |
| **`log_SOUTHAMORIGIN`** | `-0.204610` | **Moderate Negative:** South American origin percentages shift predictions slightly downward under multivariate control. |
| **`log_MEXICANORIGIN`** | `-0.138868` | **Small Negative:** Holds a mild negative corrective effect. |
| **`log_RENTERS`** | `+0.137586` | **Small Positive:** Higher renter shares lightly increase the likelihood of a county matching this profile. |
| **`log_CARIBBEAN`** | `-0.118767` | **Small Negative:** Holds a minor negative effect on the classification. |
| **`INTPTLON`** | `+0.096417` | **Weak Positive:** Western longitude position holds a very minor positive impact on the classification. |
| **`VOTELEAN`** | `-0.038028` | **Near Zero:** Local conservative-leaning voting blocks exert a negligible negative influence on the profile. |
| **`log_MEDINCOME`** | `-0.020373` | **Near Zero:** Retained an almost entirely muted negative weight, showing minimal linear contribution when other socioeconomics are held constant. |

---

#### 4. Comparison: Univariate vs. Multivariate

Comparing the 14 isolated, univariate models against the simultaneous, ML models shows that some variables capture/correlate with the effects of others.

##### The Education Paradox (`log_BACHDEG`)

* Univariate Impact: The coefficient was 0.2261, meaning that an increase in this by 1 unit multiplies the probability of SOR being chosen by 1.25. 
* Multivariate Impact: However, in the predictive model, it has a extremely negative value, being the largest overall weight.
* In isolation, counties with more bachelor's degrees appear positively associated with higher `Most_SOR` likelihoods. However, once population density, income, and immigration metrics are simultaneously controlled by the pipeline (which are high in highly educated areas), higher educational attainment flips to act as the single strongest negative anchor against single-race SOR classification. 


##### Non-Citizenship and Language Validation

* Unlike the features that flip direction, language access barriers and citizenship status carry a robust, structurally consistent positive predictive signal that scales cleanly from independent environments straight into highly controlled multivariate matrices.
* However, the country-of-origin matters. 
    * When we only look at one variable at a time, `MEXICANORIGIN` (1.40), `CENTRALAMORIGIN` (1.40), and `SOUTHAMORIGIN` (1.18) all operate as highly significant, positive independent predictors.
    * But in the multivariate predicting model, only `CENTRALAMORIGIN` remains positive (0.635723), while `MEXICANORIGIN` (-0.138868) and `SOUTHAMORIGIN` (-0.204610) flip to negative weights.
* In isolation, larger concentrations of these specific heritage groups signal a higher baseline likelihood of meeting the SOR threshold. Inside the combined model, however, once the general immigrant and language vectors absorb the broader trend, Central American concentrations emerge as the unique positive differentiator driving single-race "Some Other Race" selection, forcing other ancestral variables into negative corrective roles.


#### 5. Feature Importance: Random Forest Classifier

These tell us the predictive power. The scores add up to 1, so the higher the magnitude, the more useful the variable was in determining the classification. 

| Feature Name | Importance Score | Relative Significance & Role |
| :--- | :---: | :--- |
| **`log_SPANISHLIMENGLISH`** | `0.155074` |Reaffirms the logistic regression finding. |
| **`log_MEXICANORIGIN`** | `0.100223` |  Ranks second overall, revealing a massive divergence from the multivariate logistic model where it was indicated to have minor influence. This indicates that while Mexican ancestry may be redundant once other factors are controlled, it is a massive structural differentiator. |
| **`log_CENTRALAMORIGIN`** | `0.098299` |  Confirms country-of-origin context as an essential feature. |
| **`log_NONCITIZENS`** | `0.085742` | Functions as a critical predictor. |
| **`log_BACHDEG`** | `0.076906` |  Confirms that educational attainment is highly informative, regardless of whether its linear relationship flips. |
| **`MEDAGE`** | `0.074569` | Age distribution provides consistent utility for mid-level node splits. |
| **`INTPTLON`** | `0.059553` |  Longitude plays a noticeably larger role in the tree architecture than it did in the linear model, indicating geographic regionality. |
| **`log_FOREIGNBORN`** | `0.053022` | Acts as a helpful baseline contextual feature for broader immigration patterns. |
| **`log_RENTERS`** | `0.052071` | Housing dynamics offer steady, localized predictive power. |
| **`log_MEDINCOME`** | `0.047983` |  Holds minor independent utility, likely diluted by correlations with other more important (in the sense of this model) variables. |
| **`VOTELEAN`** | `0.044136` |  Political leaning remains a secondary, minor descriptor. |
| **`log_DENSITY`** | `0.042543` |  Density provides minor utility when structural demographic features are already present. |
| **`log_NOINTERNET`** | `0.038703` | Limited contribution. |
| **`log_CARIBBEAN`** | `0.038446` |  Holds minimal contribution. |
| **`log_SOUTHAMORIGIN`** | `0.032731` |  Contributes the least amount to our prediction. |



In [8]:
# 1. Reconstruct feature dataframe based on your lists
X_features = census_and_geodata[vars_no_log].copy()

for col in vars_log:
    # Using log1p (log(1+x)) keeps 0 entries safe from evaluating to negative infinity
    X_features[f'log_{col}'] = np.log1p(census_and_geodata[col])

# Choose your target column (e.g., 'Most_SOR')
y_target = census_and_geodata['Most_White_SOR']

# Combine to perform an explicit dropna alignment across rows
model_data = pd.concat([X_features, y_target], axis=1).dropna()

# Final clean feature matrix and target vector
X = model_data.drop(columns=['Most_White_SOR'])
y = model_data['Most_White_SOR'].astype(int)

# Train-test split (80/20 ratio matching organ-followup standard)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Calculate naive mean probability from training target
mean_prob = y_train.mean()
y_train_baseline = np.full_like(y_train, mean_prob, dtype=float)
y_test_baseline = np.full_like(y_test, mean_prob, dtype=float)

# Determine global majority class for accuracy baseline
majority_class = 1 if mean_prob >= 0.5 else 0

print("--- Baseline Naive Model ---")
print(f"Train Log Loss: {log_loss(y_train, y_train_baseline):.4f}")
print(f"Test Log Loss:  {log_loss(y_test, y_test_baseline):.4f}")
print(f"Train Accuracy: {accuracy_score(y_train, np.full_like(y_train, majority_class)):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, np.full_like(y_test, majority_class)):.4f}")

# Build a pipeline to scale data uniformly across folds
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr_cv', LogisticRegressionCV(penalty='l1', solver='saga', cv=5, max_iter=5000, random_state=42))
])

# Train model
lr_pipeline.fit(X_train, y_train)

print("--- Regularized Logistic Regression (L1-CV) ---")
print(f"Train Log Loss: {log_loss(y_train, lr_pipeline.predict_proba(X_train)):.4f}")
print(f"Test Log Loss:  {log_loss(y_test, lr_pipeline.predict_proba(X_test)):.4f}")
print(f"Train Accuracy: {accuracy_score(y_train, lr_pipeline.predict(X_train)):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, lr_pipeline.predict(X_test)):.4f}")


# Initialize Random Forest (max_depth restricted to mitigate severe overfitting)
rf_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf_model.fit(X_train, y_train)

print("--- Random Forest Classifier ---")
print(f"Train Log Loss: {log_loss(y_train, rf_model.predict_proba(X_train)):.4f}")
print(f"Test Log Loss:  {log_loss(y_test, rf_model.predict_proba(X_test)):.4f}")
print(f"Train Accuracy: {accuracy_score(y_train, rf_model.predict(X_train)):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, rf_model.predict(X_test)):.4f}")

# Create a DataFrame matching features to coefficients for the Logistic model
importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': lr_pipeline.named_steps['lr_cv'].coef_[0]
})

# Sort by absolute magnitude to see the most influential features at the top
importance_df['Abs_Magnitude'] = importance_df['Coefficient'].abs()
importance_df = importance_df.sort_values(by='Abs_Magnitude', ascending=False).drop(columns=['Abs_Magnitude'])

# Display the clean dataframe
print(importance_df.reset_index(drop=True))

# Create a DataFrame matching features to coefficients, for the Random Forest
rf_importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_model.feature_importances_
})

rf_importance_df = rf_importance_df.sort_values(
    by='Importance',
    ascending=False
).reset_index(drop=True)

print('--- Random Forest importance dataframe ---')
rf_importance_df


/Users/ireland/Desktop/Spring2026/.venv/lib/python3.11/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/ireland/Desktop/Spring2026/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/Users/ireland/Desktop/Spring2026/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1811: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' instead. Use l1_ratios=(0,) instead of penalty='l2'  and l1_ratios=(1,) instead of penalty='l1'.
  warnings.warn(
/Users/ire

--- Baseline Naive Model ---
Train Log Loss: 0.3896
Test Log Loss:  0.3361
Train Accuracy: 0.8683
Test Accuracy:  0.8967
--- Regularized Logistic Regression (L1-CV) ---
Train Log Loss: 0.3355
Test Log Loss:  0.3110
Train Accuracy: 0.8727
Test Accuracy:  0.8903
--- Random Forest Classifier ---
Train Log Loss: 0.2073
Test Log Loss:  0.2684
Train Accuracy: 0.9260
Test Accuracy:  0.9062
                  Feature  Coefficient
0             log_BACHDEG     1.708374
1             log_RENTERS    -1.449969
2         log_FOREIGNBORN     0.759589
3     log_CENTRALAMORIGIN    -0.491965
4          log_NOINTERNET    -0.421632
5                  MEDAGE     0.260323
6           log_MEDINCOME     0.247086
7             log_DENSITY    -0.207226
8       log_MEXICANORIGIN    -0.183643
9   log_SPANISHLIMENGLISH    -0.124693
10      log_SOUTHAMORIGIN     0.122464
11               VOTELEAN    -0.119375
12          log_CARIBBEAN     0.086497
13               INTPTLON     0.006034
14        log_NONCITIZENS    

,Feature,Importance
0,log_MEDINCOME,0.164913
1,MEDAGE,0.091613
2,INTPTLON,0.075447
3,VOTELEAN,0.068104
4,log_BACHDEG,0.067050
5,log_DENSITY,0.063985
6,log_RENTERS,0.062683
7,log_MEXICANORIGIN,0.058637
8,log_SPANISHLIMENGLISH,0.058195
9,log_NOINTERNET,0.058088


### Predictive Model Evaluation & Feature Analysis: White & Some Other Race (`Most_White_SOR`)

We then evaluated models targeting `Most_White_SOR`. The target variable features a training mean of 86.83%, which serves as a highly imbalanced baseline. Consequently, a baseline naive model achieves a high classification accuracy by default.

#### 1. Performance Metrics Summary

| Model | Train Log Loss | Test Log Loss | Train Accuracy | Test Accuracy |
| :--- | :---: | :---: | :---: | :---: |
| **Baseline Naive Model** | 0.3896 | 0.3361 | 86.83% | 89.67% |
| **Regularized Logistic Regression ($L_1$-CV)** | 0.3355 | 0.3110 | 87.27% | 89.03% |
| **Random Forest Classifier** | 0.2073 | 0.2684 | 92.60% | 90.62% |

---

#### 2. Key Findings & Performance Interpretation

* Because 86.83% of the training instances fall into the majority class, the Baseline Naive model sets an exceptionally high bar for accuracy. 
* While the regularized Logistic Regression model shows slightly lower test accuracy, it substantially improves the Log Loss. This indicates that the logistic pipeline produces significantly more nuanced, calibrated, and confident probability metrics than naive guess.
* The Random Forest Classifier achieved the highest predictive performance on the test set. It also minimized test Log Loss. This demonstrates that tree-based architectures are successful as they were for `Most_SOR`.

---

#### 3. Feature Importance & Coefficient Analysis for Logistic Regression


| Feature Name | Coefficient Value | Impact Direction on `Most_White_SOR` |
| :--- | :---: | :--- |
| **`log_BACHDEG`** | `+1.708374` | **Very Strong Positive:** The dominant driver in the multivariate model. Higher rates of college graduation strongly increase a county's likelihood of being classified as `Most_White_SOR` when controlling for other socioeconomic factors. Note the difference in sign from `Most_SOR`. |
| **`log_RENTERS`** | `-1.449969` | **Very Strong Negative:** Higher proportions of renters heavily decrease the odds of this classification under multivariate control. |
| **`log_FOREIGNBORN`** | `+0.759589` | **Strong Positive:** Higher overall concentrations of foreign-born residents serve as a prominent positive marker for this group profile. |
| **`log_CENTRALAMORIGIN`** | `-0.491965` |  **Moderate-Strong Negative:** Higher shares of Central American ancestry act as a structural negative qualifier. |
| **`log_NOINTERNET`** | `-0.421632` |  **Moderate Negative:** Counties with severe digital infrastructure deficits correlate negatively with meeting this demographic threshold. |
| **`MEDAGE`** | `+0.260323` |  **Moderate Positive:** Older county median ages shift linear predictions upward. |
| **`log_MEDINCOME`** | `+0.247086` | **Moderate Positive:** Solid income levels lightly increase the marginal probability of matching this demographic profile. |
| **`log_DENSITY`** | `-0.207226` |**Moderate Negative:** Dense, urban population centers moderately suppress the odds within the pipeline. |
| **`log_MEXICANORIGIN`** | `-0.183643` | **Small-Moderate Negative:** Higher concentrations of Mexican ancestry exert a slight negative pressure. |
| **`log_SPANISHLIMENGLISH`** | `-0.124693` | **Small Negative:** Higher rates of limited English proficiency pull the multivariate model slightly away from the threshold. |
| **`log_SOUTHAMORIGIN`** | `+0.122464` | **Small Positive:** South American origin layers minor positive predictive pressure into the model. |
| **`VOTELEAN`** | `-0.119375` | **Small Negative:** Higher local conservative-leaning voting blocks decrease the likelihood of matching this target profile. |
| **`log_CARIBBEAN`** | `+0.086497` |  **Weak Positive:** Holds a negligible positive impact on classification. |
| **`INTPTLON`** | `+0.006034` |  **Near Zero:** Retained a near-zero positive weight, meaning geographic longitude has virtually no contribution. |
| **`log_NONCITIZENS`** | `0.000000` | **Zeroed Out:** Completely eliminated by the Lasso penalty as a predictive redundancy. |

---

#### 4. Comparison: Univariate vs. Multivariate

Comparing the 14 isolated, univariate models against the simultaneous, ML models shows that some variables capture/correlate with the effects of others. However, these variables are different than in the `Most_SOR` case. 

##### The Renter Inversion Paradox (`log_RENTERS`)
* Univariate: This has a negligable contribution, with an increase in this value by 1 multiplying the probability of White_SOR by 1.
* Multivariable: In constrast, in the multivariate predictor model, this variable has an extremely negative relationship with this race selection, and is in fact the second largest weight. 
* Once the model simultaneously controls for localized population density, median income, and immigration metrics, renter share emerges as an incredibly powerful negative anchor. This highlights that homeownership configurations provide a heavy structural signal that is completely masked in univariate assessments.

##### The Education Amplification (`log_BACHDEG`)
* Univariate: Multiplies probability by factor of 1.14, which is mild. 
* Multivariate: This is the strongest driver of the predictive model, with a coefficient of 1.708374
* Independent evaluations show that higher educational attainment holds a statistically significant, moderate positive correlation with the target. Within the multivariate model, its effect is dramatically amplified to become the single strongest predictor. The multivariate predictor model is anchored by the bachelor's degree rates.

##### The Muting of Non-Citizenship (`log_NONCITIZENS`)
* In both models, non-citizenship is not a strong predictor. Especially as its coefficient is zeroed out in our multivariate predicting model, the other variables are collinear and adding it to the model is redundant. 
* As we get into specific origin, however, we see the weights are significant. 
    * Univariate: `MEXICANORIGIN` is independently negative, with a multiplicative factor of 0.94. Meanwhile, `SOUTHAMORIGIN` is positive, at a multiplicative factor of 1.13. `CENTRALAMORIGIN` is statistically insignificant.
    * Multivariate: `CENTRALAMORIGIN` now has a moderate negative weight (-0.491965), `MEXICANORIGIN` becomes more negative (-0.183643), and `SOUTHAMORIGIN` remains a small positive contribution (0.122464).  
    * In isolation, Central American populations appear completely unrelated to the outcome; but in the multivariate model, it flips to become a prominent negative benchmark. This indicates that after controlling for aggregate immigrant patterns, specific regional subgroups function as important spatial and identity course-corrections inside the controlled pipeline.

#### 5. Feature Importance: Random Forest Classifier 


| Feature Name | Importance Score | Relative Significance & Role |
| :--- | :---: | :--- |
| **`log_MEDINCOME`** | `0.164913` | A massive departure from the `Most_SOR` model. Median household income skyrockets to the single most critical lens the model uses to partition the data, heavily outperforming all other variables. |
| **`MEDAGE`** | `0.091613` |  The age distribution of a county provides significant utility for this model. |
| **`INTPTLON`** | `0.075447` |  Longitude plays a prominent role here, signaling that this specific profile has a distinct East-versus-West geographic footprint across the country. |
| **`VOTELEAN`** | `0.068104` | Unlike the pure SOR model where local politics exerted almost zero influence, political composition is elevated to a frontline predictive feature. |
| **`log_BACHDEG`** | `0.067050` | Like in `Most_SOR`'s Random Forest Classifier model, this variable is middle-tier in its utility for tree partitioning. |
| **`log_DENSITY`** | `0.063985` | Offers moderate predictive power. |
| **`log_RENTERS`** | `0.062683` | Offers moderate predictive power. |
| **`log_MEXICANORIGIN`** | `0.058637` | Suffers a significant drop from its top-tier status in the `Most_SOR` Random Forest Classifer model. |
| **`log_SPANISHLIMENGLISH`** | `0.058195` | In comparison to its utility in `Most_SOR`, this variable has the biggest drop in importance. |
| **`log_NOINTERNET`** | `0.058088` | Offers minor predictive power. |
| **`log_CENTRALAMORIGIN`** | `0.049598` | Offers minor predictive power. |
| **`log_SOUTHAMORIGIN`** | `0.047056` | Offers minor predictive power.|
| **`log_CARIBBEAN`** | `0.046822` | Offers minor predictive power. |
| **`log_FOREIGNBORN`** | `0.045118` | Offers minor predictive power. |
| **`log_NONCITIZENS`** | `0.042693` | Offers minor predictive power.|

---


Comparing this model to the previous `Most_SOR` pipeline reveals a complete paradigm shift in what defines the classification boundary. 

* The `Most_SOR` classification was driven by culture and immigration status: language access barriers (`log_SPANISHLIMENGLISH`) and specific ancestral origin vectors (`log_MEXICANORIGIN`, `log_CENTRALAMORIGIN`).
* The `Most_White_Sor` classification was driven by class, age, and geography. The tree relies on a macro baseline of affluence (`log_MEDINCOME`), generational architecture (`MEDAGE`), regional landscape (`INTPTLON`), and political ecosystem (`VOTELEAN`). It focuses heavily on the broader, institutional socioeconomic realities of the counties involved.

In [9]:
# 1. Reconstruct feature dataframe based on your lists
X_features = census_and_geodata[vars_no_log].copy()

for col in vars_log:
    # Using log1p (log(1+x)) keeps 0 entries safe from evaluating to negative infinity
    X_features[f'log_{col}'] = np.log1p(census_and_geodata[col])

# Choose your target column (e.g., 'Most_SOR')
y_target = census_and_geodata['Most_White']

# Combine to perform an explicit dropna alignment across rows
model_data = pd.concat([X_features, y_target], axis=1).dropna()

# Final clean feature matrix and target vector
X = model_data.drop(columns=['Most_White'])
y = model_data['Most_White'].astype(int)

# Train-test split (80/20 ratio matching organ-followup standard)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Calculate naive mean probability from training target
mean_prob = y_train.mean()
y_train_baseline = np.full_like(y_train, mean_prob, dtype=float)
y_test_baseline = np.full_like(y_test, mean_prob, dtype=float)

# Determine global majority class for accuracy baseline
majority_class = 1 if mean_prob >= 0.5 else 0

print("--- Baseline Naive Model ---")
print(f"Train Log Loss: {log_loss(y_train, y_train_baseline):.4f}")
print(f"Test Log Loss:  {log_loss(y_test, y_test_baseline):.4f}")
print(f"Train Accuracy: {accuracy_score(y_train, np.full_like(y_train, majority_class)):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, np.full_like(y_test, majority_class)):.4f}")

# Build a pipeline to scale data uniformly across folds
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr_cv', LogisticRegressionCV(penalty='l1', solver='saga', cv=5, max_iter=5000, random_state=42))
])

# Train model
lr_pipeline.fit(X_train, y_train)

print("--- Regularized Logistic Regression (L1-CV) ---")
print(f"Train Log Loss: {log_loss(y_train, lr_pipeline.predict_proba(X_train)):.4f}")
print(f"Test Log Loss:  {log_loss(y_test, lr_pipeline.predict_proba(X_test)):.4f}")
print(f"Train Accuracy: {accuracy_score(y_train, lr_pipeline.predict(X_train)):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, lr_pipeline.predict(X_test)):.4f}")


# Initialize Random Forest (max_depth restricted to mitigate severe overfitting)
rf_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf_model.fit(X_train, y_train)

print("--- Random Forest Classifier ---")
print(f"Train Log Loss: {log_loss(y_train, rf_model.predict_proba(X_train)):.4f}")
print(f"Test Log Loss:  {log_loss(y_test, rf_model.predict_proba(X_test)):.4f}")
print(f"Train Accuracy: {accuracy_score(y_train, rf_model.predict(X_train)):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, rf_model.predict(X_test)):.4f}")

# Create a DataFrame matching features to coefficients for the Logistic model
importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': lr_pipeline.named_steps['lr_cv'].coef_[0]
})

# Sort by absolute magnitude to see the most influential features at the top
importance_df['Abs_Magnitude'] = importance_df['Coefficient'].abs()
importance_df = importance_df.sort_values(by='Abs_Magnitude', ascending=False).drop(columns=['Abs_Magnitude'])

# Display the clean dataframe
print(importance_df.reset_index(drop=True))

# Create a DataFrame matching features to coefficients, for the Random Forest
rf_importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_model.feature_importances_
})

rf_importance_df = rf_importance_df.sort_values(
    by='Importance',
    ascending=False
).reset_index(drop=True)

print('--- Random Forest importance dataframe ---')
rf_importance_df

/Users/ireland/Desktop/Spring2026/.venv/lib/python3.11/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/ireland/Desktop/Spring2026/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/Users/ireland/Desktop/Spring2026/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1811: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' instead. Use l1_ratios=(0,) instead of penalty='l2'  and l1_ratios=(1,) instead of penalty='l1'.
  warnings.warn(
/Users/ire

--- Baseline Naive Model ---
Train Log Loss: 0.6366
Test Log Loss:  0.6138
Train Accuracy: 0.6665
Test Accuracy:  0.6995
--- Regularized Logistic Regression (L1-CV) ---
Train Log Loss: 0.5165
Test Log Loss:  0.4789
Train Accuracy: 0.7286
Test Accuracy:  0.7456
--- Random Forest Classifier ---
Train Log Loss: 0.3456
Test Log Loss:  0.4467
Train Accuracy: 0.8997
Test Accuracy:  0.7758
                  Feature  Coefficient
0             log_DENSITY    -3.947061
1             log_RENTERS     0.839245
2   log_SPANISHLIMENGLISH    -0.832855
3         log_NONCITIZENS    -0.742023
4     log_CENTRALAMORIGIN    -0.443934
5       log_MEXICANORIGIN     0.273502
6             log_BACHDEG     0.221982
7                VOTELEAN     0.204181
8           log_MEDINCOME    -0.149696
9           log_CARIBBEAN     0.130226
10      log_SOUTHAMORIGIN     0.126249
11               INTPTLON    -0.125989
12                 MEDAGE     0.082915
13        log_FOREIGNBORN    -0.054283
14         log_NOINTERNET    

,Feature,Importance
0,log_SPANISHLIMENGLISH,0.131913
1,log_NONCITIZENS,0.107872
2,log_CENTRALAMORIGIN,0.104236
3,log_MEXICANORIGIN,0.093718
4,log_FOREIGNBORN,0.065405
5,log_RENTERS,0.063551
6,log_BACHDEG,0.055780
7,VOTELEAN,0.052192
8,INTPTLON,0.052035
9,log_NOINTERNET,0.049969


### Predictive Model Evaluation & Feature Analysis: White Alone (`Most_White`)

Following our predictive modeling methods, we evaluated models targeting `Most_White`. The target variable features a training mean of 66.65%, indicating a moderate class imbalance where the majority of counties fall into this group profile. 

#### 1. Performance Metrics Summary

| Model | Train Log Loss | Test Log Loss | Train Accuracy | Test Accuracy |
| :--- | :---: | :---: | :---: | :---: |
| **Baseline Naive Model** | 0.6366 | 0.6138 | 66.65% | 79.95% |
| **Regularized Logistic Regression ($L_1$-CV)** | 0.5165 | 0.4789 | 72.86% | 74.56% |
| **Random Forest Classifier** | 0.3456 | 0.4467 | 89.97% | 77.58% |

---

#### 2. Key Findings & Performance Interpretation

* High Baseline Threshold: Due to the 66.65% balance in the training set, the naive model sets a high benchmark for classification accuracy on the test set. 
* Logistic regression: Although strict binary classification thresholds place the Logistic Regression's test accuracy slightly below the naive guess, it provides an improvement in Log Loss. This proves that the regularized linear pipeline is successfully nuanced. 
* Random Forest: As with the other race variables, the Random Forest Classifier produced good results, reducing Log Loss even more. Its test accuracy, was still lower than that of the niave model, however. Moroever, difference of its training to test accuracy does suggest overfitting.

---

#### 3. Feature Importance & Coefficient Analysis of Logistic Regression



| Feature Name | Coefficient Value | Impact Direction on `Most_White` |
| :--- | :---: | :--- |
| **`log_DENSITY`** | `-3.947061` | **Extremely Strong Negative:** The dominant feature in the model. Higher population density drastically drops the odds of a county being classified as `Most_White`. |
| **`log_RENTERS`** | `+0.839245` |  **Strong Positive:** Higher proportions of renters serve as a powerful positive predictor in this simultaneous model environment. |
| **`log_SPANISHLIMENGLISH`** | `-0.832855` | **Strong Negative:** Higher shares of limited English proficiency households act as a major negative anchor pulling the model away from the threshold. |
| **`log_NONCITIZENS`** | `-0.742023` | **Strong Negative:** Higher percentages of non-citizen residents heavily suppress the likelihood of this classification. |
| **`log_CENTRALAMORIGIN`** | `-0.443934` | **Moderate Negative:** Concentrations of Central American ancestry function as a reliable negative qualifier. |
| **`log_MEXICANORIGIN`** | `+0.273502` | **Moderate Positive:** Holds a positive predictive weight. |
| **`log_BACHDEG`** | `+0.221982` |  **Moderate Positive:** Higher levels of college graduation rates push the county-level prediction upward. |
| **`VOTELEAN`** | `+0.204181` | **Moderate Positive:** Local conservative-leaning voting blocks shift the pipeline positively toward the this race. |
| **`log_MEDINCOME`** | `-0.149696` |  **Small-Moderate Negative:** Higher median household income exerts a minor downward pressure. |
| **`log_CARIBBEAN`** | `+0.130226` | **Small Positive:** Percentages of Caribbean ancestry layer a minor positive weight into the matrix. |
| **`log_SOUTHAMORIGIN`** | `+0.126249` | **Small Positive:** South American origin concentrations shift the marginal prediction slightly upward. |
| **`INTPTLON`** | `-0.125989` |  **Small Negative:** A negative longitude coefficient shows a minor linear shift toward the Eastern US. |
| **`MEDAGE`** | `+0.082915` | **Weak Positive:** Older median county age provides a very small upward predictive push. |
| **`log_FOREIGNBORN`** | `-0.054283` |  **Near Zero:** Negligible impact. |
| **`log_NOINTERNET`** | `-0.042981` | **Near Zero:** Negligible impact. |

---

#### 4. Synthesis: Univariate vs Multivariate

Comparing our 14 isolated, univariate models against the simultaneous, standardized machine learning methods show the following:

##### Opposite Renter variable behavior (`log_RENTERS`)
* Univariate: this has a negative impact in isolation, multiplying the probability of `Most_White` by 0.70. 
* Multivariate: However, our predictive model that combines all our variables assign the log(renters) variable a highly positive coefficient. It is the second largest overall weight. 
* Alone, counties with higher proportions of renters appear strongly negatively correlated with `Most_White` status (capturing general urban vs. rural). However, once population density, income, and immigration are simultaneously isolated and controlled by the pipeline, higher renter shares completely invert to become a powerful positive driver.

##### Opposite Mexican origin variable behavior (`log_MEXICANORIGIN`)
* Univariate: this has a negative impact in isolation, multiplying the probability of `Most_White` by 0.70. 
* Multivariate: Meanwhile, in our logistic predictive model, this variable has a moderately positive contribution (0.273502) 
* In isolation, higher concentrations of Mexican ancestry display a negative relationship with the target. Yet inside the combined model, once the system adjusts for specific indicators like citizenship status (`log_NONCITIZENS`) and language barriers (`log_SPANISHLIMENGLISH`), the isolated effect of Mexican ancestry flips positive.

##### Opposite Mexican origin variable behavior (`log_BACHDEG`)
* Univariate:  this has a negative impact in isolation, multiplying the probability of `Most_White` by 0.69. `coef = -0.3724` ($z = -13.76$, $p < 0.001$) [cite: 12]
* Multivariate: Again, in our logisistic predictive model, this variable had a moderately positive contribution (0.221982)
* In independent models, higher college graduation rates appear to pull heavily away from `Most_White` status. Once the pipeline controls for population density, higher educational attainment reveals its true positive predictive relationship with this target profile.

##### Consistencies:
* Univariate: Population density, limited English, and non-citizenship all yield highly significant, negative independent coefficients.
* Multivariate: All three remain prominent, powerful negative weights in the combined machine learning system.

#### 5. Feature Importance: Random Forest Classifier (`Most_White`)



| Feature Name | Importance Score | Relative Significance & Role |
| :--- | :---: | :--- |
| **`log_SPANISHLIMENGLISH`** | `0.131913` | Once again ranks as the single most powerful feature. Language access patterns remain the most efficient tool the trees have for partitioning demographic majorities. |
| **`log_NONCITIZENS`** | `0.107872` | Functions as a heavy-lifting variable. |
| **`log_CENTRALAMORIGIN`** | `0.104236` | Ranks remarkably high, serving as a critical indicator. |
| **`log_MEXICANORIGIN`** | `0.093718` | Maintains a powerful presence in utility, confirming that regional heritage densities are highly informative to the model. |
| **`log_FOREIGNBORN`** | `0.065405` | Offers moderate predictive power. |
| **`log_RENTERS`** | `0.063551` | Offers moderate predictive power. |
| **`log_BACHDEG`** | `0.055780` | Offers moderate predictive power.|
| **`VOTELEAN`** | `0.052192` | Offers moderate predictive power. |
| **`INTPTLON`** | `0.052035` |Offers moderate predictive power. |
| **`log_NOINTERNET`** | `0.049969` | Digital infrastructure gaps offer limited marginal utility to the trees. |
| **`log_MEDINCOME`** | `0.046573` | In stark contrast to the wealth-driven `Most_White_Sor` model, direct household income drops significantly in predictive priority here. |
| **`log_DENSITY`** | `0.046256` | Offers minor predictive power. |
| **`MEDAGE`** | `0.046235` | Offers minor predictive power. |
| **`log_CARIBBEAN`** | `0.045936` | Offers minor predictive power. |
| **`log_SOUTHAMORIGIN`** | `0.038330` | Offers minor predictive power. |

---


Via the lens of the Random Forest Classifier, comparing the feature importances of `Most_White` against the previous models reveals a fascinating structural alignment:

* When predicting the `Most_White` classification boundary, the Random Forest model reverts to a feature hierarchy that looks similar to the original `Most_SOR` model, prioritizing linguistic and demographic features over pure economic ones.  In both cases, language barriers (`log_SPANISHLIMENGLISH`), citizenship status (`log_NONCITIZENS`), and Central American/Mexican heritage metrics dominate the top spots, while wealth (`log_MEDINCOME`) and age (`MEDAGE`) sit near the bottom. 
* Meanwhile, the `Most_White_SOR` sub-profile was defined by variables relating to wealth, age, and geography, and behaved much differently.


## Comparisons
* For all three targets, the Random Forest model achieved the lowest test Log Loss.
* In the cases of `Most_White_SOR` and `Most_White`, the strict binary classification accuracy of the naive baseline appears deceptively high due to class imbalances. However, the machine learning models (both Logistic Regression and Random Forest) still outperform the baseline.

---

### Summaries


1. `Most_SOR` is driven by language, immigration, and origin:
   * Both the logistic model and Random Forest feature importances prioritize linguistic barriers (`log_SPANISHLIMENGLISH`), citizenship status (`log_NONCITIZENS`), and specific country-of-origin concentrations (particularly Central American and Mexican heritage metrics). 
2. `Most_White_SOR` is driven by class/wealth:
   * In comparison to `Most_SOR`, the tree-based model for `Most_White_SOR` drops cultural and immigration indicators to the bottom. Instead, the boundary is heavily defined by macro indicators of affluence (`log_MEDINCOME`), generational architecture (`MEDAGE`), housing configurations (`log_RENTERS`), and local political landscapes (`VOTELEAN`).
3. `Most_White` is driven by density in one model, but language, immigration, and origin in another:
   * In the Logistic predictive model, population density (`log_DENSITY`) emerges as the single most significant contributor, meaning that urban concentration heavily drives a county away from this classification. When using the Random Forest, the model reverts to a linguistic and demographic feature hierarchy similar to `Most_SOR`, utilizing language and citizenship as efficient indicators for splitting the data.
4. In our analysis of `Most_SOR`, we see that higher rates of college graduation strongly decrease a county's likelihood of matching this profile when other variables are controlled. In `Most_White_SOR`, it acts as a strong positive contributor to classification. The impact on `Most_White`, while positive, is moderate once population density is taken into account. 
5. The number of renters in a county also varies the direction of its contribution; in predicting `Most_SOR`, this variable yields a mild positive effect on our prediction model. In predicting `Most_White_SOR`, it functions as as a negative contributor, indicating that heavy renter shares strongly pull a county away from this racial choice. Finally, in predicting `Most_White`, the variable's behavior contrasts with the other two choices by being a strong positive predictor. 

# Continuous predictors

In [5]:
census_and_geodata['HWHITEACOMBO'] = census_and_geodata['WHITEALONEORCOMBO'] - census_and_geodata['NONHISPANICWHITEALONEORCOMBO']
census_and_geodata['Hisp White PL Percent'] = census_and_geodata['HWHITEACOMBO']/census_and_geodata['TOTALPOP']
census_and_geodata['H_N_WHITEACOMBO'] = census_and_geodata['HISPANIC'] - census_and_geodata['HWHITEACOMBO']

## Random Forest Regressor

What does this do?

1. We have four target variables, representing the different race combinations. 
    * `Hisp SOR Alone PL Percent` 
    * `Hisp White SOR PL Percent` 
    * `Hisp White Alone PL Percent` 
    * `Hisp White PL Percent` (analyzing this will explain why Hispanic/Latino pick White)
2. Then we loop through each target to build a model for it.
3. For each specific target variable, we: split the dataset into Training ($80\%$) and Testing ($20\%$) subsets using `train_test_split` with a fixed `random_state=42` to guarantee reproducible results across runs.
4. The instance of `RandomForestRegressor` is initialized with `n_estimators=100` (100 individual decision trees) and restricted to a `max_depth=8` to prevent overfitting. 
5. After training, we get the relative predictive power of each variable using `rf_regressor.feature_importances_`, and rank the features from most impactful to least impactful. This is saved in a dictionary (`rf_target_importances`). 

In [6]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegressionCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss, accuracy_score


target_proportions = ['Hisp SOR Alone PL Percent', 'Hisp White SOR PL Percent', 'Hisp White Alone PL Percent', 'Hisp White PL Percent']

# Reconstruct feature dataframe based on your lists
X_features = census_and_geodata[vars_no_log].copy()
for col in vars_log:
    # Using log1p (log(1+x)) keeps 0 entries safe from evaluating to negative infinity
    X_features[f'log_{col}'] = np.log1p(census_and_geodata[col])

# Dictionary to store the importance DataFrames for later inspection
rf_target_importances = {}

for target in target_proportions:
    print(f"Training Random Forest Regressor for target: {target}...")
    
    model_data = pd.concat([X_features, census_and_geodata[target]], axis=1).dropna()
    
    X_t = model_data.drop(columns=[target])
    y_t = model_data[target]
    
    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(X_t, y_t, test_size=0.2, random_state=42)
    
    # Train the model using Training data
    rf_regressor = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42) # you want to prevent overfitting, but changing max_depth would be interesting
    rf_regressor.fit(X_train, y_train)
    
    # Generate predictions on the unseen test features
    y_pred = rf_regressor.predict(X_test)
    
    # Evaluate with test data
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    
    print(f"[{target}] R² Score: {r2:.4f} | RMSE: {rmse:.4f}")
    # =================================================
    
    # Create the target-specific importance dataframe
    importance_df = pd.DataFrame({
        'Feature': X_train.columns,
        'Importance': rf_regressor.feature_importances_
    }).sort_values(by='Importance', ascending=False).reset_index(drop=True)
    
    rf_target_importances[target] = importance_df

# Print dataframes
print("\n--- Top Features for Predicting SORA Proportions ---")
print(rf_target_importances['Hisp SOR Alone PL Percent'])

print("\n--- Top Features for Predicting 'White + Some Other Race' Proportions ---")
print(rf_target_importances['Hisp White SOR PL Percent'])

print("\n--- Top Features for Predicting White A Proportions ---")
print(rf_target_importances['Hisp White Alone PL Percent'])

print("\n--- Top Features for Predicting White Alone or in Combination Proportions ---")
print(rf_target_importances['Hisp White PL Percent'])



/Users/ireland/Desktop/Spring2026/.venv/lib/python3.11/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


Training Random Forest Regressor for target: Hisp SOR Alone PL Percent...
[Hisp SOR Alone PL Percent] R² Score: 0.8464 | RMSE: 0.0217
Training Random Forest Regressor for target: Hisp White SOR PL Percent...
[Hisp White SOR PL Percent] R² Score: 0.7861 | RMSE: 0.0209
Training Random Forest Regressor for target: Hisp White Alone PL Percent...
[Hisp White Alone PL Percent] R² Score: 0.8099 | RMSE: 0.0170
Training Random Forest Regressor for target: Hisp White PL Percent...
[Hisp White PL Percent] R² Score: 0.8453 | RMSE: 0.0332

--- Top Features for Predicting SORA Proportions ---
                  Feature  Importance
0       log_MEXICANORIGIN    0.385976
1             log_BACHDEG    0.154659
2   log_SPANISHLIMENGLISH    0.108071
3                INTPTLON    0.099794
4             log_RENTERS    0.054448
5                  MEDAGE    0.051686
6          log_NOINTERNET    0.043674
7             log_DENSITY    0.024184
8     log_CENTRALAMORIGIN    0.019403
9         log_NONCITIZENS    0.013

## Log-Ratio Regression Pipeline

This code creates log-ratio outcome variables from census racial/ethnic percentage data and then applies two regularized regression methods — Ridge Regression and Lasso Regression — to identify which features are associated with different Hispanic racial identification patterns.

We define census proportion variables and add epsilon, as we will be dividing and don't want division by 0. The code transforms the proportions into log-ratios: $\log\left(\frac{A}{B}\right)$. The ratios we look at are as follows: 
1. $\log\left(\frac{\text{Some Other Race Alone}}{\text{Any White Identification}}\right)$; higher values indicate stronger identification with Some Other Race rather than White.
2. $\log\left(\frac{\text{White + Some Other Race}}{\text{Any White Identification}}\right)$; this captures the tendency for respondents who identify as White to also select Some Other Race.
3. $\log\left(\frac{\text{White Alone}}{\text{Any White Identification}}\right)$; this measures the tendency toward monoracial or exclusive White identification.

We use two methods to manage multicollinearity:
1. Ridge regression is a regularized linear regression method. The loss function is an L^2 measure. Again, we split the data into training and test. The model tests 100 different penalty strengths, $10^{-3} \text{ to } 10^{3}$. Then cross-validation automatically selects the penalty (`alpha`) that best manages model fit. Features are standardized to have mean 1 and standard deviation 0 because the model is sensitive to variable scale. Then the model learns the relationship between our independent variables and our log-ratio. The coefficients tell us the strength of the association like above, but also gives us the direction as well. 
2. Lasso regression uses L^1 loss measure and can shrink coefficients all the way to zero. We test a variety of penalty strengths to find the relevant parameters. 

In [11]:
# Add a tiny epsilon to handle any rare absolute zero values safely
epsilon = 1e-5
p_sora = census_and_geodata['Hisp SOR Alone PL Percent'] + epsilon
p_w_sor = census_and_geodata['Hisp White SOR PL Percent'] + epsilon
p_whitea = census_and_geodata['Hisp White Alone PL Percent'] + epsilon

# Your new baseline anchor (the broad umbrella category)
p_white_total = census_and_geodata['Hisp White PL Percent'] + epsilon 

# Compute Log-Ratios directly without summing to 1
# This perfectly handles the subset nature of the data
census_and_geodata['y_1_SORA_vs_WhiteTotal'] = np.log(p_sora / p_white_total)
# This measures the people who chose SOR and did not choose White at all, against those that did choose White.
census_and_geodata['y_2_WhiteSOR_vs_WhiteTotal'] = np.log(p_w_sor / p_white_total)
# This measures who chose White + SOR in comparison to who chose White at all (so we see odds that a person also picked sor)
census_and_geodata['y_3_WhiteA_vs_WhiteTotal'] = np.log(p_whitea / p_white_total)
# This measures who chose White in comparison to who chose White at all (so odds that they only view themselves as white/monoracial)

# Update your target list for the regressions
log_ratio_targets = ['y_1_SORA_vs_WhiteTotal', 'y_2_WhiteSOR_vs_WhiteTotal', 'y_3_WhiteA_vs_WhiteTotal']


from sklearn.linear_model import RidgeCV  # Ridge handles multi-collinearity well

for target_name in log_ratio_targets:
    print(f"\n==================================================")
    print(f"RUNNING REGULARIZED LINEAR REGRESSION FOR: {target_name}")
    print(f"==================================================")
    
    # Align features and target
    model_data = pd.concat([X_features, census_and_geodata[target_name]], axis=1).dropna()
    X_lr = model_data.drop(columns=[target_name])
    y_lr = model_data[target_name]
    
    X_train, X_test, y_train, y_test = train_test_split(X_lr, y_lr, test_size=0.2, random_state=42)
    
    # Standardize features
    ridge_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('ridge_cv', RidgeCV(alphas=np.logspace(-3, 3, 100)))  # Tests 100 different penalty strengths; this is the machine learning part
    ])
    
    # Train
    ridge_pipeline.fit(X_train, y_train)
    
    # Extract coefficients
    coefficients = ridge_pipeline.named_steps['ridge_cv'].coef_
    
    # Dataframe formatting
    coef_df = pd.DataFrame({
        'Feature': X_train.columns,
        'Coefficient': coefficients
    })
    coef_df['Abs_Magnitude'] = coef_df['Coefficient'].abs()
    coef_df = coef_df.sort_values(by='Abs_Magnitude', ascending=False).drop(columns=['Abs_Magnitude'])
    
    print(coef_df.reset_index(drop=True))

from sklearn.linear_model import LassoCV


log_ratio_targets = ['y_1_SORA_vs_WhiteTotal', 'y_2_WhiteSOR_vs_WhiteTotal', 'y_3_WhiteA_vs_WhiteTotal']

for target_name in log_ratio_targets:
    print(f"\n==================================================")
    print(f"RUNNING LASSO REGRESSION FOR: {target_name}")
    print(f"==================================================")
    
    # Align features and target
    model_data = pd.concat([X_features, census_and_geodata[target_name]], axis=1).dropna()
    X_lr = model_data.drop(columns=[target_name])
    y_lr = model_data[target_name]
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_lr, y_lr, test_size=0.2, random_state=42
    )
    
    # Standardize features and tune alpha using cross-validation
    lasso_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('lasso_cv', LassoCV(
            alphas=np.logspace(-3, 1, 100),  # Range of penalty strengths
            cv=5,                            # 5-fold cross-validation
            random_state=42,
            max_iter=10000
        ))
    ])
    
    # Train
    lasso_pipeline.fit(X_train, y_train)
    
    # Best alpha selected by CV
    best_alpha = lasso_pipeline.named_steps['lasso_cv'].alpha_
    print(f"Best alpha: {best_alpha}")
    
    # Extract coefficients
    coefficients = lasso_pipeline.named_steps['lasso_cv'].coef_
    
    # Format and display coefficients
    coef_df = pd.DataFrame({
        'Feature': X_train.columns,
        'Coefficient': coefficients
    })
    
    # Optional: remove zeroed-out coefficients
    coef_df = coef_df[coef_df['Coefficient'] != 0]
    
    coef_df['Abs_Magnitude'] = coef_df['Coefficient'].abs()
    coef_df = (
        coef_df
        .sort_values(by='Abs_Magnitude', ascending=False)
        .drop(columns=['Abs_Magnitude'])
    )
    
    print(coef_df.reset_index(drop=True))


RUNNING REGULARIZED LINEAR REGRESSION FOR: y_1_SORA_vs_WhiteTotal
                  Feature  Coefficient
0          log_NOINTERNET     0.469495
1             log_BACHDEG    -0.383246
2         log_NONCITIZENS     0.336621
3             log_RENTERS    -0.266231
4       log_MEXICANORIGIN     0.212914
5     log_CENTRALAMORIGIN     0.143983
6                INTPTLON     0.137510
7   log_SPANISHLIMENGLISH     0.129352
8       log_SOUTHAMORIGIN    -0.123559
9                  MEDAGE    -0.119830
10        log_FOREIGNBORN    -0.119553
11          log_CARIBBEAN    -0.100970
12          log_MEDINCOME     0.041118
13               VOTELEAN    -0.035091
14            log_DENSITY     0.013435

RUNNING REGULARIZED LINEAR REGRESSION FOR: y_2_WhiteSOR_vs_WhiteTotal
                  Feature  Coefficient
0             log_RENTERS    -0.306758
1         log_FOREIGNBORN     0.282209
2             log_BACHDEG     0.171496
3       log_MEXICANORIGIN     0.135997
4          log_NOINTERNET     0.134301
5   